In [1]:
using Gridap.ODEs
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields
using Random
using LinearAlgebra
using Gridap.Arrays
using Base.Iterators
using LineSearches: BackTracking

In [2]:
const E = 32e9 
const ν = 0.2 
const G = E/(2*(1+ν))
const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = G
G0 = 2.7e-7 
const fₜ = 6.0e6
const κ = λ + μ
r = 0.4e-3 
const E_0_dot1 = 3.0e-6
const c1 = 0.055
ρ = 2450

2450

In [3]:
model = GmshDiscreteModel("SingleEdgeNotch_mod.msh")
writevtk(model,"SingleEdgeNotch_mod")

Info    : Reading 'SingleEdgeNotch_mod.msh'...
Info    : 19 entities
Info    : 23705 nodes
Info    : 47412 elements
Info    : Done reading 'SingleEdgeNotch_mod.msh'


3-element Vector{Vector{String}}:
 ["SingleEdgeNotch_mod_0.vtu"]
 ["SingleEdgeNotch_mod_1.vtu"]
 ["SingleEdgeNotch_mod_2.vtu"]

In [4]:
order = 1
reffeG = ReferenceFE(lagrangian,VectorValue{1,Float64},order)
VG = TestFESpace(model,reffeG;
          conformity=:H1)

order = 1
reffephi  = ReferenceFE(lagrangian,Float64,order)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)
Uphi = TrialFESpace(Vphi)
f_new = 1.0

1.0

In [5]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V = TestFESpace(model,reffe;
          conformity=:H1)

UnconstrainedFESpace()

In [6]:
Ω = Triangulation(model)
degree = 2*order
dΩ = Measure(Ω,degree)

GenericMeasure()

In [7]:
labels = get_face_labeling(model)
LoadTagId = get_tag_from_name(labels,"BottomEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)
LoadTagId_top = get_tag_from_name(labels,"TopEdge")
Γ_Load_top = BoundaryTriangulation(model,tags = LoadTagId_top)
dΓ_Load_top = Measure(Γ_Load_top,degree)
n_Γ_Load_top = get_normal_vector(Γ_Load_top)

GenericCellField():
 num_cells: 22
 DomainStyle: ReferenceDomain()
 Triangulation: BoundaryTriangulation()
 Triangulation id: 152386584479455046

In [8]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

47263-element Table{Int32, Vector{Int32}, Vector{Int32}}:
 [22418, 22519, 22520]
 [17566, 22086, 23328]
 [4807, 4809, 4811]
 [13233, 22204, 22603]
 [816, 919, 1609]
 [1489, 1490, 22714]
 [2714, 2911, 22696]
 [5391, 5408, 5410]
 [2873, 4354, 22532]
 [1119, 1471, 1641]
 [989, 1075, 1503]
 [4248, 13503, 13505]
 [22966, 22967, 22968]
 ⋮
 [14647, 14715, 23496]
 [14687, 14689, 14696]
 [17013, 17928, 22357]
 [17332, 17335, 17413]
 [17337, 17339, 17387]
 [17359, 17478, 20672]
 [17579, 17582, 22678]
 [17631, 22332, 22904]
 [18117, 18118, 23104]
 [22472, 22518, 22519]
 [22761, 22822, 22823]
 [22967, 22968, 22975]

In [9]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

CellPoint()

In [10]:
using NearestNeighbors
data = zeros(2,Nₙ)
data[1,:] =nodeCoordX
data[2,:] =nodeCoordY
points = data
balltree = BallTree(data)
idxs = inrange(balltree, points, r, true)

23705-element Vector{Vector{Int64}}:
 [1]
 [2]
 [3]
 [4]
 [5]
 [6, 118, 119, 120, 121, 232, 465, 466, 467, 468, 871, 875, 1092, 1453, 1480, 1496, 23640]
 [7, 120, 121, 122, 123, 232, 249, 250, 251, 252, 428, 1086, 1454, 1459, 23314, 23431, 23640, 23687]
 [8]
 [9]
 [10]
 [11]
 [12]
 [13]
 ⋮
 [1566, 1568, 1569, 1570, 23406, 23635, 23694]
 [23695]
 [17338, 17340, 22506, 22507, 23696]
 [23697]
 [125, 126, 127, 159, 183, 965, 966, 1032, 1033, 1101, 1428, 1445, 1451, 1639, 23168, 23698]
 [23699]
 [826, 862, 863, 1058, 23054, 23700]
 [23701]
 [23702]
 [2657, 2700, 2712, 2913, 2959, 2960, 4347, 22787, 23059, 23412, 23577, 23665, 23703, 23704]
 [2656, 2657, 2658, 2700, 23059, 23412, 23577, 23703, 23704]
 [14717, 14718, 22572, 23088, 23705]

In [11]:
function G_kill(σ_eq,dot_σ_eq,fₜ)
 G_kill = 0.25*G0.*(( (σ_eq./fₜ -1)+ abs∘(σ_eq./fₜ - 1))).*((dot_σ_eq)+ abs∘(dot_σ_eq))
    return G_kill
end

G_kill (generic function with 1 method)

In [12]:
function new_EnergyState(ψPlusPrev_in,t_s_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
         tPlus_out = t_s_in
        damaged = false
    else
        ψPlus_out = ψPlus_in
        tPlus_out = T
        damaged = true
    end
    damaged, ψPlus_out, tPlus_out   
end

new_EnergyState (generic function with 1 method)

In [13]:
σ(ε_nl) =  λ*tr(ε_nl)*one(ε_nl) + 2*μ*ε_nl

σ (generic function with 1 method)

In [14]:
function σ_eq(ε, ε_nl)
    εArray, εArray_nl = get_array.((ε, ε_nl))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    Λpos = diagm(0 => max.(0, Λ))
    Λpos_nl = diagm(0 => max.(0, Λ_nl))
    εPos = TensorValue(P * Λpos * P')
    εPos_nl = TensorValue(P_nl * Λpos_nl * P_nl')
    ψPos = 0.5 * ((tr(ε_nl) >= 0) * (λ * tr(ε_nl) * tr(ε_nl)) + 2μ * (εPos_nl ⊙ εPos_nl))
    return √(2 * ψPos * E)
end

σ_eq (generic function with 1 method)

In [15]:
function nonLocalGk(G_k_prev,t_k_prev)
    GkVec = evaluate(G_k_prev,x_S)
    TkVec = evaluate(t_k_prev,x_S)
    caches = [array_cache(GkVec) for k in 1:Threads.nthreads()]
    caches_T = [array_cache(TkVec) for k in 1:Threads.nthreads()]
    Gk_nds = zeros(Nₙ)
    Tk_nds = zeros(Nₙ)
    Gk_sum = zeros(Nₙ)
   Tk_sum = zeros(Nₙ)
    Gk_count = zeros(Int, Nₙ)
    Threads.@threads for iel in 1:Nₑ
        cache = caches[Threads.threadid()]
        cache_T = caches_T[Threads.threadid()]
        ElNdInd = elem[iel]
    val_G = getindex!(cache, GkVec, iel)
    val_T = getindex!(cache_T, TkVec, iel)
for node in ElNdInd
        Gk_sum[node] += val_G[1]
        Tk_sum[node] += val_T[1]
        Gk_count[node] += 1
        end
    end
Gk_nds = Gk_sum ./ Gk_count
Tk_nds = Tk_sum ./ Gk_count
  Gk_nds_NL = zeros(Nₙ)
    Threads.@threads for nd_id in 1:Nₙ
        NeighHood = idxs[nd_id]
        @inbounds Gk_nds_NL[nd_id] = (sum(Gk_nds[NeighHood]) ) / (length(NeighHood))
    end
    return Gk_nds_NL,Tk_nds
end

nonLocalGk (generic function with 1 method)

In [16]:
function nonlocalfield(Gk_nds, t)
    ϕVec_st = similar(Gk_nds)  
    @. ϕVec_st = exp(-(Gk_nds * t)) 
    return FEFunction(Uphi, ϕVec_st)
end

nonlocalfield (generic function with 1 method)

In [17]:
function E_dot(ε_new,ε_old)
    εArray, εArray_nl = get_array.((ε_new,ε_old))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    e1 = Λ[1]
    e2 = Λ[2]
    e_nl1 = Λ_nl[1]
    e_nl2 = Λ_nl[2]
     Λ_new = max(e1,e2)
     Λ_old = max(e_nl1,e_nl2)
    E_dot = (1/T)*(Λ_new-Λ_old)
end

E_dot (generic function with 1 method)

In [18]:
function RateDepGc(_E_dot)
    if _E_dot > E_0_dot1
     ft2 = fₜ*(1+((_E_dot/E_0_dot1)^c1 - 1 ))
    else 
         ft2 = fₜ
    end
    return  ft2
end

RateDepGc (generic function with 1 method)

## constant average acceleration method

In [19]:
function step_disp(uApp,dot_σ_eq,uh,vh,ah,G_k_cell,ϕ,dt,t_cell,dot_E,uh_upd,G_k_prev,trac_top,trac)
           nls = NLSolver(
  show_trace=true, method=:newton, iterations = 10, linesearch=BackTracking(),)
solver = FESolver(nls)  
U = TrialFESpace(V)
σ_eq_s = σ_eq∘((ε(uh_upd))*ϕ,(ε(uh_upd))*ϕ)
ft_new= fₜ 
G_k_in = G_kill(σ_eq_s,dot_σ_eq,ft_new)
update_state!(new_EnergyState,G_k_cell,t_cell,G_k_in)
G_k_nd_nl,t_nds = nonLocalGk(G_k_cell,t_cell)
ϕ = (nonlocalfield(G_k_nd_nl,t_nds))
    m(t, u, v) = ∫(ρ*v⋅(∂t(∂t(u))))dΩ
      a(t,u,v) = ∫( (ε(v)) ⊙ ((σ∘((ε(u))*(ϕ)))*(ϕ)))*dΩ
l_disp(t,v) =  ∫(trac⋅ v) * dΓ_Load + ∫(trac_top⋅ v) * dΓ_Load_top

delT=dt
res(t, u, v) = m(t, u, v) + a(t, u, v) - l_disp(t, v) 
    ls = LUSolver()
nl_solver = NLSolver(ls, method=:newton, iterations=10, show_trace=true)

order = 2
op = TransientFEOperator(res,U,V;order)
t0 = T-dt
tF = T
γ = 0.5
β = 0.25   
ode_solver = Newmark(nl_solver,dt,γ,β)
sol_t = solve(ode_solver,op,t0,tF,(uh,vh,ah))
u_new = zero(V)
        for (tn,uhn) in sol_t
            u_new = uhn
        end
a_vec = (4/(delT*delT))*(get_free_dof_values(u_new)-get_free_dof_values(uh)-delT*get_free_dof_values(vh))-get_free_dof_values(ah)
a_new = FEFunction(Ua,a_vec)
    v_vec = get_free_dof_values(vh) + 0.5*delT*(get_free_dof_values(ah)+get_free_dof_values(a_new))  
v_new = FEFunction(Uv,v_vec)
return u_new,v_new,a_new, ϕ,G_k_cell,t_cell, G_k_nd_nl, ft_new, t_nds, cache
end

step_disp (generic function with 1 method)

In [20]:
cd("SENT_local_New_energy_30_06_2025_inbuilt")

In [21]:
ft_new =interpolate_everywhere(fₜ,Vphi)
G_k_cell = CellState(0.0,dΩ_ro)
T= 0.0
# Energy=0.0
g= 0.0 #0.0 #9.8
t_cell = CellState(T,dΩ_ro)
uApp = 0.0
vApp = 0.1
delT = 1e-7
Tmax = 1e-4
Load = Float64[]
Displacement = Float64[]
 Energy = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)
push!(Energy, 0.0)
trac_top = VectorValue(0.0,1.0e6)
trac = VectorValue(0.0,-1.0e6)
count_n = 0
count_inner = 0
uh = zero(V)
dt=delT
uh_in_FE = zero(V)
 uh_iter = zero(V)
uh_iter_p = zero(V)
vh_iter_cell = CellState(VectorValue(0.0,0.0),dΩ)
ah_iter_cell = CellState(VectorValue(0.0,0.0),dΩ)
v1(x) = VectorValue(0.0,0.0)
v2(x) = VectorValue(0.0,-vApp) #VectorValue(0.0,-vApp) # VectorValue(0.0,0.0)
v3(x) = VectorValue(0.0,0.0)
a2(x) =VectorValue(0.0,-g) 
Uv = TrialFESpace(V)
Ua = TrialFESpace(V)
vh_vec = get_free_dof_values(uh)
vh_iter = FEFunction(Uv,vh_vec)
vh_iter_p = vh_iter
ah_vec = get_free_dof_values(uh)
ah_iter = FEFunction(Ua,ah_vec)
ah_iter_p = ah_iter
dot_σ_eq = (σ_eq∘(ε(uh),(ε(uh))) - σ_eq∘(ε(uh_in_FE),(ε(uh_in_FE))))./delT
G_k_cell = CellState(0.0,dΩ_ro)
G_k_prev = zero(Vphi)
innerMax = 20
dot_E = E_dot∘(ε(uh),ε(uh_in_FE))
G_k_nd_nl,t_nds = nonLocalGk(G_k_cell,t_cell)
ϕ = interpolate_everywhere(f_new,Uphi)
σ_eq_s = σ_eq∘(ε(uh),(ε(uh)))
cache = nothing
while T < Tmax 
        T = T + delT
    count_n = count_n +1
    # vApp = 17.4e-3 #g*T
    uApp  = T*vApp
    uh_iter = uh
        dot_E = E_dot∘(ε(uh_iter),ε(uh_iter_p))
    print("\n Entering displacemtent step :", float(uApp)) 
    for inner = 1:innerMax
   
    uh,vh,ah,ϕ,G_k_cell,t_cell,G_k_nd_nl,ft_new,t_nds,cache =  step_disp(uApp,dot_σ_eq,uh_iter,vh_iter,ah_iter,G_k_cell,ϕ,dt,t_cell,dot_E,uh,G_k_prev,trac_top,trac)     
strain_energy_density = ε(uh) ⊙ (σ∘((ε(uh)) * ϕ))
        dot_σ_eq = (σ_eq∘(ε(uh)*ϕ,(ε(uh))*(ϕ)))./T
    e = uh-uh_in_FE      
    err = sqrt(sum( ∫( e⊙e )*dΩ ))
    print("\n Error :", float(err))  
    uh_in_FE = uh
    vh_iter = vh
    ah_iter = ah
    if err <=1e-8
        break
    end
    end
    Node_Force = sum(∫( n_Γ_Load ⋅ ((σ∘( (ε(uh))*ϕ))*ϕ )) *dΓ_Load)
    push!(Load, abs(Node_Force[2]))
    push!(Displacement, T)
    if mod(count_n,25) == 0
   writevtk(Ω,"SENT_results_$count_n",cellfields=["disp"=>uh,"phi"=>ϕ,"sig"=>σ_eq∘(ε(uh),(ε(uh))*ϕ) ,"ah"=>ah_iter,"vh"=>vh_iter])  
    end
end"


 Entering displacemtent step :1.0e-8Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     4.855676e+03              NaN
     1     5.911716e-12     1.122136e+13

 Error :1.4471320270234995e-11
 Entering displacemtent step :2.0e-8Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     6.660507e+02              NaN
     1     1.818989e-12     6.758785e+11

 Error :5.642004100313504e-11
 Entering displacemtent step :3.0e-8Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     1.012849e+03              NaN
     1     1.364242e-12     1.624234e+12

 Error :1.0760373668695871e-10
 Entering displacemtent step :4.0e-8Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     1.128496e+03              NaN
     1     3.183231e-12     1.651020e+12

 Error :1.5047073092169486e-10
 Entering displacemtent step :5.0e-8Iter     f(x) inf-norm    Step 2-norm 
-----

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://D:/OneDrive - Indian Institute of Science/Gopal/Blast_induced_fracture/Impact_local_working_directory/SENT_local_New_energy_30_06_2025_inbuilt/In[21]#78:4\[90mIn[21]:78:4[0;0m]8;;\
    end
end[48;2;120;70;70m"[0;0m
[90m#  ╙ ── [0;0m[91mextra tokens after end of expression[0;0m